# Regression Trees, Random Forest and Gradient Boosting – Fuel Consumption of New Cars

**Dataset:** Fuel consumption of 701 new car models of model year 2025 (Natural Resources Canada, Open Government
Licence – Canada): make, model, vehicle class, engine size, cylinders, transmission, fuel type and fuel consumption.
File: `../00_Data/car_fuel_consumption_2025.csv`.

**Business question:** How much fuel (L/100 km) does a new car model use, depending on its engine, vehicle class,
transmission and fuel type? The features interact (a hybrid car with a continuously variable transmission uses little
fuel almost independently of its engine size, a conventional car uses much more with a larger engine) – a typical use
case for tree-based models.

**After this exercise you can ...**
- find the best split of a regression tree by hand (SSE criterion),
- fit, visualise and prune a regression tree,
- explain and apply bagging, random forests (incl. out-of-bag error, permutation importance) and gradient boosting,
- compare models fairly with cross-validation.

## Concept

The concepts behind this exercise (formulas and worked examples) are
explained in the notebook [regression_trees_ensembles_concept.ipynb](regression_trees_ensembles_concept.ipynb). Read it before you start.

## Libraries and settings

In [1]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    KFold,
)
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Ignore warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

/workspaces/python_machine_learning_basics/03_Regression_Trees_Ensembles


## Exercise 1: Calculations by hand

*Time: about 15 minutes. Use the notebook cell below as your calculator: type the formulas with the numbers in
Python (e.g. `(3 + 5) / 2` or `0.4 ** 2`) – no paper needed.*

**a) A split of a regression tree.** Six cars with the following engine sizes and fuel consumptions:

| engine size $x$ [litres] | 1.5 | 2.0 | 2.5 | 3.5 | 5.0 | 6.2 |
|:---|:---|:---|:---|:---|:---|:---|
| fuel consumption $y$ [L/100 km] | 6.5 | 7.5 | 8.5 | 11.0 | 13.0 | 14.0 |

The tree considers the split *engine size ≤ 3 litres*: left node {6.5, 7.5, 8.5}, right node {11.0, 13.0, 14.0}.
Compute the SSE of the root node (all six cars) and of both child nodes (the function `sse()` is given). By how much
does the split reduce the SSE? What are the leaf predictions (means of the two nodes)?

**b) Variance of an average.** 100 trees have variance $\sigma^2 = 4$ each. Compute the variance of their average
$\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ for $\rho = 0.6$ (bagged trees, similar to each other) and for $\rho = 0.2$
(random forest).

**c) One boosting step.** The current prediction for a car is 9.0 L/100 km, the observed consumption is 11.0. The next
tree predicts the residual exactly. What is the new prediction with learning rate $\nu = 0.1$?

In [ ]:
# a) Fuel consumption (L/100 km) of six cars, sorted by engine size
y = np.array([6.5, 7.5, 8.5, 11.0, 13.0, 14.0])
left = np.array([6.5, 7.5, 8.5])  # engine size <= 3 litres
right = np.array([11.0, 13.0, 14.0])  # engine size > 3 litres


# Function: computes the sum of squared deviations of v from its mean
def sse(v):
    """Sum of squared errors around the mean."""
    return np.sum((v - v.mean()) ** 2)


# SSE of the root and of both children, leaf predictions (means)
print('SSE root:', round(sse(y), 2))
print('SSE left:', round(sse(left), 2), ' SSE right:', round(sse(right), 2))
print('SSE after the split:', round(sse(left) + sse(right), 2))
print('Leaf predictions:', round(left.mean(), 2), round(right.mean(), 2))

# b) Variance of the average of B = 100 trees with variance 4
print('rho = 0.6:', round(0.6 * 4 + 0.4 / 100 * 4, 3))
print('rho = 0.2:', round(0.2 * 4 + 0.8 / 100 * 4, 3))

# c) One boosting step: prediction 9.0, observed 11.0, learning rate 0.1
print('New prediction:', 9.0 + 0.1 * (11.0 - 9.0))

**Solution:**

a) Root: mean $60.5/6 \approx 10.08$ L/100 km, SSE ≈ 46.71. Left node {6.5, 7.5, 8.5}: mean 7.5, SSE = 2.0; right node
{11.0, 13.0, 14.0}: mean 12.67, SSE ≈ 4.67. After the split the SSE is 6.67 – a reduction of about **86 %**. Leaf
predictions: 7.5 L/100 km for small engines (up to 3 litres) and 12.67 L/100 km for large engines.

b) $\rho = 0.6$: $0.6\cdot 4 + 0.4/100\cdot 4 = 2.416$; $\rho = 0.2$: $0.2\cdot 4 + 0.8/100\cdot 4 = 0.832$. The less
similar the trees ($\rho$ small), the smaller the variance of the average – this is why the random forest
de-correlates the trees.

c) Residual $11.0 - 9.0 = 2.0$ → new prediction $9.0 + 0.1\cdot 2.0 = 9.2$ L/100 km. Each step removes only 10 % of the
remaining residual, so many trees are needed – but the model is less prone to overfitting.

## Exercise 2: Import and explore the car data

a) Import the car data. The target is the column `combined` (fuel consumption in L/100 km, combined city and
highway driving). Use the features `engine_size`, `cylinders`, `vehicle_class`, `transmission_type` and `fuel_type`
(the text columns as dummy variables: `pd.get_dummies(..., columns=[...], dtype=int)`).

b) Plot `combined` against `engine_size`, separately for cars with a continuously variable transmission and for all
other cars. What do you see?

c) Split the data into training (80 %) and test set (20 %) with `random_state=42`.

In [ ]:
# a) Import the data as cars
cars = pd.read_csv('../00_Data/car_fuel_consumption_2025.csv', sep=',')

# Features (text columns as dummy variables: one 0/1 column per category)
features = [
    'engine_size',
    'cylinders',
    'vehicle_class',
    'transmission_type',
    'fuel_type',
]
X = pd.get_dummies(
    cars[features],
    columns=['vehicle_class', 'transmission_type', 'fuel_type'],
    dtype=int,
)
y = cars['combined']
print(X.shape)
print(y.describe().round(1))

# b) Scatter plot for cars with and without continuously variable transmission
cvt = cars[cars['transmission_type'] == 'continuously variable']
other = cars[cars['transmission_type'] != 'continuously variable']
plt.scatter(
    other['engine_size'], other['combined'], label='other transmissions'
)
plt.scatter(cvt['engine_size'], cvt['combined'], label='continuously variable')
plt.xlabel('Engine size [litres]')
plt.ylabel('Fuel consumption [L/100 km]')
plt.legend()
plt.show()

# c) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print('Training set:', X_train.shape, ' Test set:', X_test.shape)

**Solution:**

a) 701 car models with 24 features (engine size, cylinders and 22 dummy variables for vehicle class, transmission type and
fuel type). The cars use on average 10.9 L/100 km (between 4.4 and 25.0 L/100 km).

b) For most cars, the fuel consumption rises clearly with the engine size (about +1.7 L/100 km per additional litre).
Cars with a **continuously variable transmission** – many of them hybrids – use much less fuel (7.4 instead of
11.5 L/100 km on average), and for them the consumption hardly rises with the engine size (+0.1 L/100 km per litre).
The effect of the engine size therefore **depends on the transmission** – such an **interaction** cannot be captured by
a linear model with additive effects, but it is natural for a tree.

## Exercise 3: Baseline – linear regression

Fit a multiple linear regression with all features on the training set and compute RMSE and $R^2$ on the test set. This
is the baseline that the tree-based models have to beat.

In [ ]:
# Exercise 3: linear regression with all features; test RMSE and R²
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)
print('Test RMSE:', round(root_mean_squared_error(y_test, y_pred), 3))
print('Test R2:', round(r2_score(y_test, y_pred), 3))

## Exercise 4: A small regression tree

a) Fit a `DecisionTreeRegressor(max_depth=3, random_state=42)` on the training set.

b) Visualise the tree with `tree.plot_tree()` and interpret the first two levels. Which features are used?

c) Compute RMSE and $R^2$ on the test set. How does the small tree compare to the linear regression?

In [ ]:
# a) Fit the tree
reg_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
reg_tree.fit(X_train, y_train)

# b) Plot the tree
plt.figure(figsize=(20, 7))
tree.plot_tree(reg_tree, feature_names=list(X_train.columns), filled=True)
plt.show()

# c) Test RMSE and R²
y_pred = reg_tree.predict(X_test)
print('Test RMSE:', round(root_mean_squared_error(y_test, y_pred), 3))
print('Test R2:', round(r2_score(y_test, y_pred), 3))

**Solution:**

The tree splits first on the **engine size** (≤ 2.6 litres). For small engines it then asks whether the car has a
**continuously variable transmission** (≈ 6.8–7.9 L/100 km with, ≈ 7.8–9.6 L/100 km without), for large engines it splits
again on the engine size (≤ 3.6 litres) and on the vehicle class (standard pickups use more fuel). One leaf contains only
the cars with 16 cylinders (≈ 25 L/100 km). The leaves range from ≈ 6.8 to ≈ 25 L/100 km.

With only 8 leaves the small tree has a test RMSE of ≈ 1.66 L/100 km ($R^2 \approx 0.68$) and is **worse** than the linear
regression (≈ 1.47, $R^2 \approx 0.75$): it is easy to interpret but has a high bias – it predicts only 8 different values.

## Exercise 5: Tree complexity and pruning

a) For `max_depth` = 1, 2, …, 15 compute the RMSE on the training set and the 5-fold CV RMSE (on the training set).
Plot both curves. From which depth on does a deeper tree no longer help?

b) Alternatively, grow a full tree and prune it: compute the candidate values with
`DecisionTreeRegressor(random_state=42).cost_complexity_pruning_path(X_train, y_train).ccp_alphas` and choose `ccp_alpha`
with `GridSearchCV` (5-fold CV, `scoring='neg_root_mean_squared_error'`). How many leaves does the best pruned tree have?
What is its test RMSE?

In [ ]:
# Exercise 5: tree complexity and pruning
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# a) Training and CV error for different depths
depths = range(1, 16)
rmse_train = []
rmse_cv = []
for d in depths:
    model = DecisionTreeRegressor(max_depth=d, random_state=42)
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error'
    )
    rmse_cv.append(-scores.mean())
    model.fit(X_train, y_train)
    rmse_train.append(root_mean_squared_error(y_train, model.predict(X_train)))

# Figure showing the training and CV error by tree depth
plt.plot(depths, rmse_train, marker='o', label='training error')
plt.plot(depths, rmse_cv, marker='o', label='5-fold CV error')
plt.xlabel('max_depth')
plt.ylabel('RMSE [L/100 km]')
plt.legend()
plt.show()
print(pd.Series(rmse_cv, index=depths).round(2))

# b) Pruning: candidate alphas of the full tree, GridSearchCV for ccp_alpha
#    (store the best value as best_alpha)

# Full tree and its candidate values of alpha
full_tree = DecisionTreeRegressor(random_state=42)
full_tree.fit(X_train, y_train)
alphas = full_tree.cost_complexity_pruning_path(X_train, y_train).ccp_alphas
print('Leaves of the full tree:', full_tree.get_n_leaves())

# Choose ccp_alpha with 5-fold CV
grid_tree = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    {'ccp_alpha': alphas},
    cv=cv,
    scoring='neg_root_mean_squared_error',
)
grid_tree.fit(X_train, y_train)
best_alpha = grid_tree.best_params_['ccp_alpha']
pruned_tree = grid_tree.best_estimator_
print('Best ccp_alpha:', round(best_alpha, 4))
print('Leaves of the pruned tree:', pruned_tree.get_n_leaves())
print('CV RMSE of the pruned tree:', round(-grid_tree.best_score_, 3))
y_pred = pruned_tree.predict(X_test)
print(
    'Pruned tree - test RMSE:',
    round(root_mean_squared_error(y_test, y_pred), 3),
)

**Solution:**

a) The training error decreases steadily with the depth and approaches 0 for deep trees. The CV error decreases until
about **depth 10** (≈ 1.25 L/100 km) and then stays the same – deeper trees no longer help. In contrast to many other
data sets the CV error hardly rises again: many car models appear in several very similar variants (e.g. with
all-wheel drive or a different transmission), so even a deep tree finds a similar car for each new car.

b) Pruning with cross-validation reduces the full tree from 207 to **65 leaves** without losing accuracy (CV RMSE ≈ 1.23,
test RMSE ≈ 1.37 L/100 km). The pruned tree is clearly better than the small tree of Exercise 4 and also better than
the linear regression.

## Exercise 6: Random forest

a) Fit random forests with `n_estimators` = 1, 5, 10, 25, 50, 100, 200, 400 (`oob_score=True`, `random_state=42`,
`n_jobs=-1`). The OOB predictions are stored in `oob_prediction_`. Plot the OOB RMSE against the number of trees. When do
the gains plateau?

b) Compare `max_features` = 1.0 (all features, i.e. bagging), `'sqrt'` and 0.5 with 5-fold CV (300 trees).

c) For the best forest, compute the test RMSE. Compare the impurity-based feature importance (`feature_importances_`)
with the **permutation importance** on the test set.

In [ ]:
# a) Number of trees vs. OOB error
n_trees = [1, 5, 10, 25, 50, 100, 200, 400]
oob_rmse = []
for n in n_trees:
    rf = RandomForestRegressor(
        n_estimators=n, oob_score=True, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    # OOB prediction of each training car (NaN if the car was never OOB)
    oob_pred = rf.oob_prediction_
    ok = ~np.isnan(oob_pred)
    oob_rmse.append(root_mean_squared_error(y_train[ok], oob_pred[ok]))
    print('n_estimators =', n, ' OOB RMSE =', round(oob_rmse[-1], 3))

# Figure showing the OOB RMSE by number of trees
plt.plot(n_trees, oob_rmse, marker='o')
plt.xscale('log')
plt.xlabel('n_estimators (log scale)')
plt.ylabel('OOB RMSE [L/100 km]')
plt.show()

# b) 5-fold CV RMSE for different values of max_features
for mf in [1.0, 'sqrt', 0.5]:
    rf = RandomForestRegressor(
        n_estimators=300, max_features=mf, random_state=42, n_jobs=-1
    )
    scores = cross_val_score(
        rf, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error'
    )
    print('max_features =', mf, ' CV RMSE =', round(-scores.mean(), 3))

In [ ]:
# c) Random forest with the best setting from b): test RMSE and feature
#    importance (impurity-based and permutation importance)

# Best setting from b): max_features=0.5
rf_best = RandomForestRegressor(
    n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1
)
rf_best.fit(X_train, y_train)
y_pred = rf_best.predict(X_test)
print('Test RMSE:', round(root_mean_squared_error(y_test, y_pred), 3))
print('Test R2:', round(r2_score(y_test, y_pred), 3))

# Impurity-based and permutation importance (on the test set)
perm = permutation_importance(
    rf_best, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
importance = pd.DataFrame(
    {
        'impurity-based': rf_best.feature_importances_,
        'permutation': perm.importances_mean,
    },
    index=X_train.columns,
)
importance = importance.sort_values('permutation')
print(importance.round(3))

# Figure comparing both importance measures
importance.plot(kind='barh', figsize=(7, 8))
plt.xlabel('importance')
plt.show()

**Solution:**

a) A forest with only one tree has a very high OOB error (≈ 8.9 L/100 km); the error drops quickly with the first 25
trees and **plateaus** at about 1.20 L/100 km from 50 trees on. More trees never hurt the accuracy, they only cost
computing time.

b) Here the value of `max_features` hardly matters (CV RMSE 1.20–1.21 for all three settings). The engine size is so
important that the trees remain similar even with random feature subsets. (Note: scikit-learn uses all features by
default for regression.)

c) The random forest reaches a test RMSE of ≈ 1.29 L/100 km ($R^2 \approx 0.81$) – clearly better than the single tree
and the linear regression. By far the most important feature is the **engine size**, followed by the **cylinders** and
the **continuously variable transmission**. The two importance measures differ for correlated features: impurity-based
importance gives the cylinders a high weight (≈ 0.25), permutation importance a much smaller one (≈ 0.14), because the
number of cylinders adds little once the engine size is known. Permutation importance is computed on the test set and is
therefore the more reliable measure.

## Exercise 7: Gradient boosting

a) Fit `GradientBoostingRegressor(n_estimators=1000, max_depth=3, random_state=42)` with learning rates
$\nu$ = 0.01, 0.1 and 1.0. Use `staged_predict(X_test)` to compute the test RMSE after each boosting iteration and plot
the three curves.

*Note: we look at the test curve here only for illustration. In practice, `learning_rate` and `n_estimators` are tuned
with cross-validation (see Exercise 8).*

b) Describe the influence of the learning rate. What happens with $\nu = 1.0$?

In [ ]:
# a) Test error after each tree for three learning rates
for lr in [0.01, 0.1, 1.0]:
    gb = GradientBoostingRegressor(
        n_estimators=1000, max_depth=3, learning_rate=lr, random_state=42
    )
    gb.fit(X_train, y_train)

    # staged_predict gives the predictions after 1, 2, ..., 1000 trees
    rmse_staged = []
    for y_pred in gb.staged_predict(X_test):
        rmse_staged.append(root_mean_squared_error(y_test, y_pred))
    plt.plot(range(1, 1001), rmse_staged, label='learning rate ' + str(lr))
    print(
        'Learning rate',
        lr,
        '- min test RMSE:',
        round(min(rmse_staged), 3),
        'after',
        np.argmin(rmse_staged) + 1,
        'trees;',
        'after 1000 trees:',
        round(rmse_staged[-1], 3),
    )

# Figure showing the test error by number of trees
plt.ylim(1, 3)
plt.xlabel('Number of trees M')
plt.ylabel('Test RMSE [L/100 km]')
plt.legend()
plt.show()

**Solution:**

- $\nu = 0.01$: very small steps – the error decreases slowly and is still decreasing after 1'000 trees (≈ 1.29).
- $\nu = 0.1$: the error decreases quickly and reaches the lowest value (≈ 1.21 after about 420 trees); this is the
  default and usually a good compromise.
- $\nu = 1.0$: each tree corrects the full residual. The error drops very fast at the beginning, reaches its minimum
  (≈ 1.33) after only about 30 trees and then **rises again** slightly (≈ 1.38 after 1'000 trees) – the model starts to
  fit the noise of the training data (overfitting).

Learning rate and number of trees must be chosen **together**: a smaller $\nu$ needs a larger $M$.

## Exercise 8: Fair model comparison

Compare the following models with **5-fold CV RMSE on the training set** and summarise the results in a table:
linear regression, pruned tree (best `ccp_alpha` from Exercise 5), random forest (best setting from Exercise 6) and
gradient boosting (`learning_rate=0.05`, `n_estimators=300`, `max_depth=3`). Then evaluate the best model **once** on the
test set. Which model would you recommend to the car importer, and what are the drawbacks?

In [ ]:
# Exercise 8: the four models (settings from Exercises 5 and 6)
models = {
    'Linear regression': LinearRegression(),
    'Pruned tree': DecisionTreeRegressor(ccp_alpha=best_alpha, random_state=42),
    'Random forest': RandomForestRegressor(
        n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1
    ),
    'Gradient boosting': GradientBoostingRegressor(
        learning_rate=0.05, n_estimators=300, max_depth=3, random_state=42
    ),
}

# 5-fold CV RMSE of all models (store them in cv_rmse)
cv_rmse = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error'
    )
    cv_rmse[name] = -scores.mean()
    print(
        name,
        '- CV RMSE:',
        round(-scores.mean(), 3),
        ' std:',
        round(scores.std(), 3),
    )

# Fit the best model and evaluate it once on the test set
# (the random forest has the smallest CV RMSE)
best_model = models['Random forest']
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(
    'Random forest - test RMSE:',
    round(root_mean_squared_error(y_test, y_pred), 3),
)
print('Random forest - test R2:', round(r2_score(y_test, y_pred), 3))

**Solution:**

| model | CV RMSE [L/100 km] |
|:---|:---|
| Random forest | ≈ 1.20 |
| Pruned tree | ≈ 1.23 |
| Gradient boosting | ≈ 1.27 |
| Linear regression | ≈ 1.40 |

The random forest is best and reaches a test RMSE of ≈ 1.29 L/100 km ($R^2 \approx 0.81$). All tree-based models are
better than the linear regression, because they capture the interactions (e.g. engine size × transmission). The
differences between the tree-based models are small compared to the standard deviation of the CV RMSE (≈ 0.12–0.14) –
with only 560 training cars they are not very reliable.

Recommendation: random forest for the prediction. Drawbacks: the model is a black box (only feature importance, no
simple formula or rules) and cannot extrapolate beyond the range of the training data (e.g. a new type of engine). If an
easily communicable model is needed, the pruned tree is a reasonable compromise: it is almost as good and consists of
rules.

Note: ensembles do not always win. For data with largely additive, roughly linear relationships (e.g. rents explained
by living area and location) a linear regression is often almost as good – always compare with cross-validation.

## Exercise 9: More covariates

So far the models used the engine, the vehicle class, the transmission and the fuel type. The data set contains more
information: the **make** of the car (39 makes, e.g. Toyota, BMW, Ford).

a) Add the make as dummy variables and split the data again (80/20, `random_state=42` – the same cars as before).

b) Compute the 5-fold CV RMSE of the linear regression, the random forest (best setting of Exercise 6) and gradient
boosting (as in Exercise 8) with the **extended** feature set and compare with Exercise 8.

c) Evaluate the random forest with the extended feature set on the test set and compute its permutation importance.
How important are the single makes?

In [ ]:
# a) Extended feature set: the make as additional dummy variables
features_more = features + ['make']
X_more = pd.get_dummies(
    cars[features_more],
    columns=['vehicle_class', 'transmission_type', 'fuel_type', 'make'],
    dtype=int,
)

# Same random_state as before -> the same training and test cars
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X_more, y, test_size=0.20, random_state=42
)
print(X_more.shape)

# b) CV RMSE with the extended feature set, compared with Exercise 8
for name in ['Linear regression', 'Random forest', 'Gradient boosting']:
    scores = cross_val_score(
        models[name],
        X2_train,
        y2_train,
        cv=cv,
        scoring='neg_root_mean_squared_error',
    )
    print(
        name,
        '- CV RMSE before:',
        round(cv_rmse[name], 3),
        ' with the make:',
        round(-scores.mean(), 3),
    )

# c) Random forest with the extended feature set: test RMSE and permutation
#    importance
rf_more = RandomForestRegressor(
    n_estimators=300, max_features=0.5, random_state=42, n_jobs=-1
)
rf_more.fit(X2_train, y2_train)
y_pred = rf_more.predict(X2_test)
print('Test RMSE:', round(root_mean_squared_error(y2_test, y_pred), 3))
print('Test R2:', round(r2_score(y2_test, y_pred), 3))

# Permutation importance on the test set
perm = permutation_importance(
    rf_more, X2_test, y2_test, n_repeats=10, random_state=42, n_jobs=-1
)
importance = pd.Series(perm.importances_mean, index=X_more.columns)
importance = importance.sort_values()
print(importance.tail(10).round(3))

# Figure showing the 10 most important features
importance.tail(10).plot(kind='barh')
plt.title('Permutation importance (test set)')
plt.show()

**Solution:**

b)
| model | CV RMSE (Exercise 8) | CV RMSE (with the make) |
|:---|:---|:---|
| linear regression | ≈ 1.40 | ≈ 1.30 |
| random forest | ≈ 1.20 | ≈ 1.09 |
| gradient boosting | ≈ 1.27 | ≈ 1.15 |

All models profit from the make (−7 % to −9 %).

c) The random forest with the make reaches a test RMSE of ≈ 1.23 L/100 km ($R^2 \approx 0.82$; before ≈ 1.29 and 0.81).
In the permutation importance, the single makes are hardly important – each make covers only a few cars (the most
important one, INEOS, is a maker of heavy off-road vehicles). Together, however, the makes explain differences in
technology between the manufacturers (e.g. many hybrids at Toyota) that engine size and transmission do not capture.

Lesson: **better covariates often help more than more tuning** – but they should describe something that the other
features do not already contain.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [12]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-02 14:19:07
Python Version: 3.11.16
-----------------------------------
